# BoneFract — Préparation de l'échantillon expérimental (15 000 images)

**Objet.** Ce notebook construit le sous-ensemble final de 15 000 images (12 000 train / 1 500 validation / 1 500 test) utilisé pour les trois expériences ConvNeXt-Tiny (ImageNet, transfert FracAtlas, framework proposé), chacune entraînée avec 3 graines (42, 123, 2024). L'échantillon est tiré **une seule fois** et reste identique pour les 9 entraînements.

Avant de figer cet échantillon, un audit du dataset a été mené. Il a révélé des problèmes de qualité qui ont conduit à **s'écarter légèrement du protocole initial** : l'échantillon est tiré dans les partitions officielles *après nettoyage*, et non directement dans les partitions brutes. Toutes les étapes sont décrites ci-dessous, dans l'ordre où elles sont exécutées.

---

## 1. Audit initial de BoneFract v1

| Contrôle | Résultat |
|---|---|
| Fichiers | 47 931 PNG + 1 classeur Excel (6 feuilles : chemins et étiquettes par partition) |
| Effectifs | 38 344 train / 4 793 valid / 4 794 test (conformes à la documentation) |
| Classes | 13 921 positives (fracture), 34 010 négatives (conformes) |
| Prévalence de fracture | 29,2 % train / 27,8 % valid / 29,1 % test |
| Régions | 7 (bras, pied, main, jambe, cuisse, bassin, poignet), prévalence très variable (≈17 % pour le bras, ≈50 % pour la cuisse) |
| Images | toutes 512 × 512, 0 illisible ; modes RGB (47 284), niveaux de gris (645), RGBA (2) → conversion RGB forcée dans le prétraitement |
| Étiquettes | déduites du nom de dossier (Positive = 1, Negative = 0) et recoupées avec l'Excel : aucune étiquette divergente (0 sur 47 930 images comparables) |
| Excel vs dossiers | mêmes effectifs par feuille (la lecture avec `header=0` avale la 1re ligne, d'où l'écart d'une ligne observé au premier audit). **Une seule exception** : le fichier `Arm_patient26944_Positive_001.png` est rangé dans `train/wrist/` sur le disque mais référencé `train/Arm/` dans l'Excel (même étiquette). Le pipeline utilise la région du dossier (poignet) |

**Identifiants patients.** Chaque identifiant patient correspond à exactement **une image**, et aucun identifiant n'est partagé entre partitions. Cette absence de chevauchement ne prouve donc **pas** l'absence de fuite : un même patient peut apparaître sous plusieurs identifiants.

---

## 2. Doublons exacts et étiquettes contradictoires (hash MD5)

- 14 263 images appartiennent à 6 997 groupes de copies strictement identiques (6 833 groupes de 2, 60 de 3, 103 de 4, 1 de 5).
- **2 420 groupes sont à cheval sur plusieurs partitions** : 26,9 % des images de validation (1 289) et 26,8 % de celles du test (1 286) ont un doublon exact dans une autre partition. Évaluer sur le test officiel brut surestimerait donc les performances.
- **1 183 groupes ont des étiquettes contradictoires** (2 509 images) : le même fichier est étiqueté à la fois fracture et non-fracture. 934 de ces groupes ont la même région (dont 614 dans la même partition), 249 ont des régions différentes. Il s'agit majoritairement de bruit d'étiquettes, non d'un simple effet de multi-étiquetage par région. La cause n'est pas établie (à vérifier dans la documentation ou auprès des auteurs).
- Les groupes dupliqués ne sont pas des images dégénérées (contraste comparable aux images non dupliquées).

**Règle appliquée :**
1. Retrait de toutes les images des groupes contradictoires : 2 509 images.
2. Pour les groupes cohérents, conservation d'**une seule copie**, dans la partition prioritaire **test > validation > train** : 5 940 copies retirées.
3. Pool après cette étape : **39 482 images** (train 30 614 / valid 4 389 / test 4 479).

---

## 3. Quasi-doublons : méthode retenue (et essai abandonné en annexe)

**Méthode retenue : corrélation de Pearson sur vignettes 32 × 32 (niveaux de gris).** Pour chaque image de validation/test, recherche du plus proche voisin parmi les autres partitions :

| Seuil de corrélation | Images valid | Images test |
|---|---|---|
| ≥ 0,999 | 33 | 41 |
| ≥ 0,99 | 41 | 49 |
| ≥ 0,98 | 66 | 70 |
| ≥ 0,95 | 704 | 715 |
| ≥ 0,90 | 2 224 | 2 239 |

L'inspection visuelle de paires par tranche montre de vraies copies (même cliché, contraste, cache ou cadrage légèrement différents) **jusqu'à environ 0,97**. Sous 0,98, les effectifs augmentent fortement et la ressemblance de cadrage entre patients distincts devient plausible. Le seuil de **0,97** a donc été retenu, de façon volontairement conservatrice (un faux positif coûte une image alors que le pool est largement suffisant).

**Règle appliquée** (687 paires inter-partitions uniques ≥ 0,97, soit 728 détections brutes car les paires valid/test sont comptées deux fois ; 58 paires ont des étiquettes différentes) :
- étiquettes différentes → retrait des **deux** images ;
- étiquettes identiques → retrait de la copie de la partition la moins prioritaire (test > validation > train).

Résultat : **547 images retirées** (test 31, valid 56, train 460). Pool final : **38 935 images** (train 30 154 / valid 4 333 / test 4 448). Vérification en float64 : corrélation maximale restante entre partitions de 0,96997 (< 0,97), et reconstruction du pool strictement identique au premier calcul.

**Essai abandonné (Annexe A) — hash perceptuel dhash 9 × 8.** Appliqué entre partitions, il retirait 787 images (712 train, 75 valid). L'inspection visuelle de paires retirées/conservées a montré qu'il s'agissait de **radiographies différentes** (par exemple une main face à un bassin, ou des genoux de patients distincts). Ce hash est trop grossier : les grands fonds noirs et les cadrages standards produisent des collisions. Le filtre a été abandonné, et l'échantillon tiré à ce stade (obsolète) a été écrasé.

---

## 4. Tirage de l'échantillon final

- Stratification par **région × classe** à l'intérieur de chaque partition officielle (les partitions ne sont ni mélangées ni reconstruites), avec allocation proportionnelle exacte (méthode du plus grand reste).
- Effectifs : **12 000 / 1 500 / 1 500**. Graine d'échantillonnage : **42** (distincte des graines d'entraînement 42, 123, 2024 ; tirage unique). La cellule de sauvegarde refuse d'écraser un échantillon déjà enregistré s'il diffère du nouveau tirage.
- Prévalence de fracture de l'échantillon : 29,2 % train / 26,8 % valid / 28,4 % test, identique à celle du pool. Écart maximal de répartition région × classe avec le pool : 0,04 point.
- Chevauchement MD5 et identifiant entre partitions : 0.

**Vérification finale (section 5).** Relecture de l'échantillon enregistré : toutes les images appartiennent au pool final, corrélation maximale entre partitions dans l'échantillon de 0,96955 (< 0,97), aucun MD5 partagé entre partitions, étiquettes identiques aux dossiers.

**Bilan :** 47 931 → 38 935 images après nettoyage (−8 996, soit 18,8 % : 2 509 contradictoires, 5 940 doublons exacts, 547 quasi-doublons), puis 15 000 images tirées.

**Fichiers produits** (Drive, dossier `Bonefract/`) : `index_bonefract.csv`, `pool_nettoye_bonefract.csv`, `images_retirees_bonefract.csv` (étapes MD5 uniquement), `paires_quasi_doublons.csv`, `echantillon_15000_bonefract.csv`, `echantillon_config.json`, `ecarts_chemins_excel_vs_dossiers.csv`, `ecarts_etiquettes_excel_vs_dossiers.csv`, `rapport_verification.json`, ainsi que le cache `vignettes32_pool.npz`.

---

## 5. Limites relevées à l'issue de l'analyse exploratoire

- L'identifiant patient ne permet pas de garantir l'indépendance patient entre partitions.
- Le détecteur de quasi-doublons (vignettes 32 × 32) peut manquer des copies fortement recadrées ou décalées, et les paires juste sous 0,97 ne sont pas traitées. Une vérification indépendante par embeddings du backbone est prévue.
- Les quasi-doublons situés à l'intérieur d'une même partition n'ont pas été retirés (pas de fuite entre partitions, mais mesures potentiellement plus corrélées).
- Le bruit d'étiquettes a été traité par retrait des cas contradictoires détectables, ce qui peut écarter des cas ambigus ; ses causes ne sont pas établies.
- Le benchmark porte sur un jeu nettoyé : les performances ne sont pas directement comparables à celles obtenues sur le test officiel brut. Une évaluation de sensibilité sur le test brut reste envisageable.

---

## Organisation du notebook

| Section | Contenu |
|---|---|
| **0. Configuration** | imports, chemins, paramètres de la méthode, extraction de l'archive, fonctions utilitaires |
| **1. Audit initial** | contenu du zip, indexation des images, effectifs et prévalences, recoupement avec l'Excel |
| **2. Doublons exacts** | empreintes MD5, analyse des groupes, politique de nettoyage, sauvegarde du pool nettoyé |
| **3. Quasi-doublons** | vignettes 32 × 32, choix du seuil, application de la règle, vérification, inspection visuelle |
| **4. Tirage** | échantillonnage stratifié et sauvegarde (sans écraser un tirage existant) |
| **5. Vérification finale** | contrôles sur l'échantillon enregistré, rapport JSON, recherche d'un patient |
| **Annexe A** | essai abandonné : hash perceptuel dhash |

## 0. Configuration

Imports, chemins, paramètres de la méthode (seuil de quasi-doublon, graine, effectifs cibles), extraction de l'archive et fonctions utilitaires partagées par toutes les sections.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import json, zipfile, shutil, hashlib, io, collections
from pathlib import Path, PurePosixPath
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- Chemins
BASE = Path("/content/drive/MyDrive/Bonefract")          # dossier de sortie sur le Drive
ZIP_PATH = BASE / "archive.zip"
LOCAL_ZIP = Path("/content/archive.zip")                 # copie locale (lecture plus rapide)
DEST = Path("/content/BoneFract")
TOP = "BoneFract A Bone Fracture Dataset"
ROOT = DEST / TOP
SAMPLE_CSV = BASE / "echantillon_15000_bonefract.csv"    # échantillon officiel (tirage unique)

# --- Paramètres de la méthode
SPLITS = ["train", "valid", "test"]
PRIO = {"test": 0, "valid": 1, "train": 2}               # plus petit = prioritaire (test > valid > train)
THR = 0.97                                               # seuil de quasi-doublon (corrélation de vignettes 32x32)
LOW = 0.95                                               # borne basse pour l'inspection des cas proches du seuil
SAMPLING_SEED = 42                                       # graine de l'échantillonnage (distincte des graines d'entraînement)
NEED = {"train": 12000, "valid": 1500, "test": 1500}     # effectifs cibles

# --- Valeurs obtenues au premier run (pour contrôle)
EXPECTED_POOL_MD5 = 39482      # pool après contradictoires + doublons exacts
EXPECTED_POOL_FINAL = 38935    # pool final après quasi-doublons
EXPECTED_REMOVED_NEAR = 547

report = {}                    # alimenté au fil des sections, exporté en JSON à la section 5

Mounted at /content/drive


In [3]:
# Extraction de l'archive (une seule fois par session)
if not ROOT.exists():
    if not LOCAL_ZIP.exists():
        shutil.copy(ZIP_PATH, LOCAL_ZIP)
    DEST.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall(DEST)
print("Racine :", ROOT, "| existe :", ROOT.exists())

Racine : /content/BoneFract/BoneFract A Bone Fracture Dataset | existe : True


In [4]:
# ─── Fonctions utilitaires ────────────────────────────────────────────────

def norm(s):
    return str(s).replace("\\", "/").strip()


def thumb(p):
    """Vignette 32x32 en niveaux de gris, centrée (la normalisation L2 est faite ensuite en float64)."""
    a = np.asarray(Image.open(p).convert("L").resize((32, 32), Image.BOX), dtype=np.float32).ravel()
    return a - a.mean()


def find_pairs(X, sp, thr_low, thr, chunk=500):
    """Paires inter-partitions UNIQUES (i<j) de corrélation >= thr_low.
    Retourne (DataFrame, nombre brut de détections >= thr, avec doublons valid/test)."""
    tgt = np.where(sp != "train")[0]
    cs, raw_thr = {}, 0
    for s in range(0, len(tgt), chunk):
        idx = tgt[s:s + chunk]
        S = X[idx] @ X.T
        S[sp[idx][:, None] == sp[None, :]] = -2.0
        raw_thr += int((S >= thr).sum())
        ii, jj = np.where(S >= thr_low)
        for a, b in zip(ii, jj):
            i, j = int(idx[a]), int(b)
            cs[(min(i, j), max(i, j))] = float(S[a, b])
    pr = pd.DataFrame([(i, j, c) for (i, j), c in cs.items()], columns=["i", "j", "corr"])
    return pr.sort_values("corr", ascending=False).reset_index(drop=True), raw_thr


def apply_rule(pairs_thr, lab, pri, n):
    """Étiquettes différentes -> retrait des deux ; sinon retrait de la copie de la partition la moins prioritaire."""
    drop = set()
    for i, j in zip(pairs_thr.i, pairs_thr.j):
        if lab[i] != lab[j]:
            drop.update((int(i), int(j)))
        else:
            drop.add(int(i) if pri[i] > pri[j] else int(j))
    keep = np.ones(n, bool)
    if drop:
        keep[list(drop)] = False
    return keep, drop


def max_cross_corr(X, sp, thr, chunk=500):
    """Corrélation maximale entre images de partitions différentes, et nombre de détections brutes >= thr."""
    tgt = np.where(sp != "train")[0]
    mx, n_over, arg = -2.0, 0, None
    for s in range(0, len(tgt), chunk):
        idx = tgt[s:s + chunk]
        S = X[idx] @ X.T
        S[sp[idx][:, None] == sp[None, :]] = -2.0
        n_over += int((S >= thr).sum())
        k = np.unravel_index(int(S.argmax()), S.shape)
        if S[k] > mx:
            mx, arg = float(S[k]), (int(idx[k[0]]), int(k[1]))
    return mx, n_over, arg


def stratified_sample(d, n, seed):
    """Échantillon stratifié région x classe, allocation proportionnelle exacte (plus grand reste)."""
    g = d.groupby(["region", "label"])
    exact = g.size() * n / len(d)
    alloc = np.floor(exact).astype(int)
    rest = n - alloc.sum()
    frac = (exact - alloc).sort_values(ascending=False)
    alloc.loc[frac.index[:rest]] += 1
    parts = [grp.sample(int(alloc.loc[k]), random_state=seed) for k, grp in g if alloc.loc[k] > 0]
    return pd.concat(parts)


def show_pairs(pr, data, title, n=6, seed=0, extra=None, head=False):
    """Affiche n paires d'images (colonnes i, j, corr de `pr`, indices dans `data`)."""
    sel = pr.head(n) if head else pr.sample(min(n, len(pr)), random_state=seed)
    if len(sel) == 0:
        print(title, ": aucune paire"); return
    fig, axes = plt.subplots(len(sel), 2, figsize=(6.5, 3.3 * len(sel)), squeeze=False)
    for r, (pid, row) in enumerate(sel.iterrows()):
        for c, k in enumerate([int(row.i), int(row.j)]):
            axes[r, c].imshow(Image.open(data["path"].iloc[k]).convert("L"), cmap="gray"); axes[r, c].axis("off")
            cap = f"#{pid} {data['split'].iloc[k]}|{data['region'].iloc[k]}|y={data['label'].iloc[k]}"
            if c == 0:
                cap += f"|r={row['corr']:.4f}" + (f"|{extra}={row[extra]:.3f}" if extra else "")
            axes[r, c].set_title(cap, fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

## 1. Audit initial de BoneFract v1

### 1.1 Contenu de l'archive

In [5]:
with zipfile.ZipFile(ZIP_PATH) as z:
    infos = [i for i in z.infolist() if not i.is_dir()]
    print("Fichiers dans le zip :", len(infos))
    print("Taille décompressée : %.2f Go" % (sum(i.file_size for i in infos) / 1e9))

    # Extensions
    print("\nExtensions :", Counter(PurePosixPath(i.filename).suffix.lower() for i in infos))

    # Nombre de fichiers par dossier (3 niveaux max)
    folders = Counter("/".join(PurePosixPath(i.filename).parts[:-1][:3]) for i in infos)
    print("\nDossiers :")
    for k, v in sorted(folders.items()):
        print(f"  {k or '(racine)'} -> {v}")

    # Fichiers non-image (annotations, readme, csv...)
    others = [i.filename for i in infos if PurePosixPath(i.filename).suffix.lower() not in {".jpg", ".jpeg", ".png"}]
    print("\nFichiers non-image :", others[:20])

    # Exemples de noms d'images
    imgs = [i.filename for i in infos if PurePosixPath(i.filename).suffix.lower() in {".jpg", ".jpeg", ".png"}]
    print("\nExemples d'images :", imgs[:8])

Fichiers dans le zip : 1
Taille décompressée : 3.95 Go

Extensions : Counter({'.zip': 1})

Dossiers :
  BoneFract A Bone Fracture Dataset -> 1

Fichiers non-image : ['BoneFract A Bone Fracture Dataset/BoneFract A Bone Fracture Dataset.zip']

Exemples d'images : []


### 1.2 Indexation des images (split / région / patient / classe)

In [6]:
# Index à partir de l'arborescence disque : split / région / patient / classe / fichier
rows, anomalies = [], []
for p in ROOT.rglob("*.png"):
    rel = p.relative_to(ROOT).parts
    if len(rel) != 5:
        anomalies.append(str(p)); continue
    split, region, patient, cls, fname = rel
    rows.append(dict(path=str(p), rel_path="/".join(rel), split=split, region_folder=region,
                     region=region.lower(), patient=patient, cls_folder=cls, file=fname))
df = pd.DataFrame(rows)
print("Images indexées :", len(df), "| chemins anormaux :", len(anomalies))
assert not anomalies, anomalies[:5]
print("Dossiers de classe :", Counter(df.cls_folder))
assert set(df.cls_folder) == {"Positive", "Negative"}, set(df.cls_folder)

df["label"] = (df.cls_folder == "Positive").astype(int)          # 1 = fracture
df["patient_key"] = df.region + "/" + df.patient
df["full_rel"] = TOP + "/" + df.rel_path                         # format des chemins de l'Excel
df["study_dir"] = TOP + "/" + df.rel_path.str.rsplit("/", n=1).str[0] + "/"

# --- Comparaison avec les chiffres annoncés
print("\nTotal:", len(df), "(annoncé 47 931)")
print("Positives:", df.label.sum(), "(annoncé 13 921) | Négatives:", (df.label == 0).sum(), "(annoncé 34 010)")
print("Par partition:", df.split.value_counts().to_dict(), "(annoncé 38 344 / 4 793 / 4 794)")
report["disque"] = {"n_images": int(len(df)), "par_partition": {k: int(v) for k, v in df.split.value_counts().items()}}

Images indexées : 0 | chemins anormaux : 0


AttributeError: 'DataFrame' object has no attribute 'cls_folder'

### 1.3 Prévalences et identifiants patients

In [ ]:
print("Prévalence fracture par partition :")
print(pd.crosstab(df.split, df.label, margins=True))
print(df.groupby("split").label.mean().round(4))
print("\nPrévalence fracture par région et partition :")
print(df.pivot_table(index="region", columns="split", values="label", aggfunc="mean").round(3))

# --- Patients : chevauchement entre partitions
sets = {s: set(df[df.split == s].patient) for s in df.split.unique()}
for a in sets:
    for b in sets:
        if a < b:
            print(f"Patients communs {a} / {b} :", len(sets[a] & sets[b]))
sk = {s: set(df[df.split == s].patient_key) for s in df.split.unique()}
for a in sk:
    for b in sk:
        if a < b:
            print(f"(région, patient) communs {a} / {b} :", len(sk[a] & sk[b]))
print("\nImages par patient :", df.groupby("patient_key").size().describe().round(2).to_dict())
print("Patients ayant à la fois images Positive et Negative :",
      (df.groupby("patient_key").label.nunique() > 1).sum())

### 1.4 Recoupement avec le classeur Excel (6 feuilles)

Effectifs par feuille, puis chemins d'images, puis étiquettes (Excel contre dossiers Positive/Negative).

In [ ]:
XLSX = next(ROOT.glob("*.xlsx"))
xl = pd.ExcelFile(XLSX)
print("Feuilles :", xl.sheet_names)
splits = SPLITS
for s in splits:
    for k in ("image_paths", "labeled_studies"):
        assert f"{s}_{k}" in xl.sheet_names, f"feuille manquante : {s}_{k}"

raw  = {sh: xl.parse(sh, header=None) for sh in xl.sheet_names}   # aucune ligne prise pour un en-tête
dflt = {sh: xl.parse(sh)              for sh in xl.sheet_names}   # lecture par défaut (header=0), comme au premier audit

disk_counts = df.split.value_counts().to_dict()
rows = []
for s in splits:
    for k in ("image_paths", "labeled_studies"):
        sh = f"{s}_{k}"
        rows.append({"feuille": sh, "dossiers": disk_counts[s],
                     "lignes (header=None)": len(raw[sh]), "lignes (header=0, défaut)": len(dflt[sh])})
cnt = pd.DataFrame(rows)
cnt["écart header=None"] = cnt["lignes (header=None)"] - cnt["dossiers"]
cnt["écart header=0"] = cnt["lignes (header=0, défaut)"] - cnt["dossiers"]
display(cnt)

print("\nContenu de la ligne avalée comme en-tête avec header=0 :")
for sh in dflt:
    print(f"  {sh}: {[str(c)[:90] for c in dflt[sh].columns]}")
report["excel_effectifs"] = cnt.to_dict(orient="records")

In [ ]:
for s in splits:
    assert raw[f"{s}_image_paths"].shape[1] == 1, f"{s}_image_paths : nombre de colonnes inattendu"
    assert raw[f"{s}_labeled_studies"].shape[1] == 2, f"{s}_labeled_studies : nombre de colonnes inattendu"

# --- 1) Chemins d'images : Excel vs disque
xl_img = pd.concat([pd.DataFrame({"full_rel": raw[f"{s}_image_paths"][0].map(norm), "split_sheet": s})
                    for s in splits], ignore_index=True)
set_disk, set_xl = set(df.full_rel), set(xl_img.full_rel)
only_disk = sorted(set_disk - set_xl)     # sur le disque, absentes de l'Excel
only_xl = sorted(set_xl - set_disk)       # dans l'Excel, absentes du disque
print("Chemins Excel (image_paths) :", len(xl_img), "| uniques :", len(set_xl))
print("Doublons de chemins dans l'Excel :", int(xl_img.full_rel.duplicated().sum()))
print("Sur le disque mais pas dans l'Excel :", len(only_disk), only_disk[:5])
print("Dans l'Excel mais pas sur le disque :", len(only_xl), only_xl[:5])
low = lambda S: {x.lower() for x in S}
print("Écarts si on ignore la casse :", len(low(set_disk) - low(set_xl)), "/", len(low(set_xl) - low(set_disk)))
xl_img["split_path"] = xl_img.full_rel.str.split("/").str[1]
print("Lignes dont la feuille ≠ partition inscrite dans le chemin :", int((xl_img.split_path != xl_img.split_sheet).sum()))

report["excel_chemins"] = {"n_excel": int(len(xl_img)), "doublons_excel": int(xl_img.full_rel.duplicated().sum()),
                           "disque_seulement": len(only_disk), "excel_seulement": len(only_xl)}

In [ ]:
# --- 2) Étiquettes : Excel (labeled_studies) vs dossiers Positive/Negative
xl_lab = pd.concat([pd.DataFrame({"study_dir": raw[f"{s}_labeled_studies"][0].map(norm),
                                  "label_raw": raw[f"{s}_labeled_studies"][1], "split_sheet": s})
                    for s in splits], ignore_index=True)
xl_lab["label_xl"] = pd.to_numeric(xl_lab.label_raw, errors="coerce")
print("Valeurs d'étiquette dans l'Excel :", xl_lab.label_raw.value_counts(dropna=False).to_dict())
assert xl_lab.label_xl.isin([0, 1]).all(), "étiquettes Excel hors {0,1}"

dups = xl_lab[xl_lab.study_dir.duplicated(keep=False)]
n_conf = int((dups.groupby("study_dir").label_xl.nunique() > 1).sum()) if len(dups) else 0
print("Études dupliquées dans l'Excel :", dups.study_dir.nunique(), "| avec étiquettes contradictoires :", n_conf)

# La ligne de l'Excel est cohérente avec elle-même ? (classe inscrite dans le chemin vs étiquette)
has_pos = xl_lab.study_dir.str.contains("/Positive/")
has_neg = xl_lab.study_dir.str.contains("/Negative/")
assert (has_pos ^ has_neg).all(), "chemin sans classe identifiable"
print("Lignes Excel où l'étiquette ≠ classe inscrite dans son propre chemin :", int((has_pos.astype(int) != xl_lab.label_xl).sum()))

# Études : disque vs Excel
study_disk = df.groupby("study_dir").agg(n_images=("file", "size"), label_folder=("label", "first")).reset_index()
print("Images par étude (disque) :", study_disk.n_images.value_counts().to_dict())
xl1 = xl_lab.drop_duplicates("study_dir")
mm = study_disk.merge(xl1[["study_dir", "label_xl", "split_sheet"]], on="study_dir", how="outer", indicator=True)
print("Études — présence :", mm["_merge"].value_counts().to_dict())

# Comparaison image par image
chk = df.merge(xl1[["study_dir", "label_xl"]], on="study_dir", how="left")
print("\nImages sans étiquette Excel :", int(chk.label_xl.isna().sum()))
print(pd.crosstab(chk.label, chk.label_xl.fillna(-1).astype(int),
                  rownames=["dossier"], colnames=["Excel (-1 = absent)"]))
mism = chk[chk.label_xl.notna() & (chk.label_xl != chk.label)]
print("Images dont l'étiquette Excel ≠ étiquette du dossier :", len(mism))

# Verdict automatique sur les trois écarts d'effectifs
ok_none = (cnt["écart header=None"] == 0).all()
ecart_h0 = cnt[cnt["écart header=0"] != 0][["feuille", "écart header=0"]]
print("\n=== VERDICT ===")
if ok_none and len(only_disk) == 0 and len(only_xl) == 0 and len(mism) == 0 and chk.label_xl.notna().all():
    print("Les 6 feuilles ont exactement les effectifs des dossiers (lecture header=None) ; mêmes chemins, mêmes étiquettes.")
    print("Les écarts d'une ligne observés au premier audit étaient dus à la lecture avec header=0 :")
    print(ecart_h0.to_string(index=False))
else:
    print("ATTENTION : des écarts subsistent. Voir les listes ci-dessus (only_disk, only_xl, mism) et le CSV sauvegardé.")

pd.DataFrame({"type": ["disque_seulement"] * len(only_disk) + ["excel_seulement"] * len(only_xl),
              "chemin": only_disk + only_xl}).to_csv(BASE / "ecarts_chemins_excel_vs_dossiers.csv", index=False)
mism[["full_rel", "label", "label_xl"]].to_csv(BASE / "ecarts_etiquettes_excel_vs_dossiers.csv", index=False)
report["excel_etiquettes"] = {"images_sans_etiquette": int(chk.label_xl.isna().sum()), "etiquettes_differentes": int(len(mism)),
                              "etudes_dupliquees": int(dups.study_dir.nunique()), "verdict_header_none_ok": bool(ok_none)}

## 2. Doublons exacts et étiquettes contradictoires (hash MD5)

### 2.1 Empreintes des images

Une seule lecture de chaque image calcule le MD5, les dimensions et le mode. Le dhash est calculé dans la même passe, uniquement pour l'essai abandonné de l'Annexe A.

In [ ]:
def inspect_image(path):
    try:
        with open(path, "rb") as f: b = f.read()
        md5 = hashlib.md5(b).hexdigest()
        im = Image.open(io.BytesIO(b)); im.load()
        w, h = im.size; mode = im.mode
        px = np.asarray(im.convert("L").resize((9, 8)), dtype=np.int16)      # dhash : utilisé seulement en Annexe A
        dh = format(int("".join("1" if x else "0" for x in (px[:, 1:] > px[:, :-1]).flatten()), 2), "016x")
        return md5, dh, w, h, mode, True
    except Exception as e:
        return None, None, None, None, str(e), False

with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(inspect_image, df.path, chunksize=64))
df[["md5", "dhash", "w", "h", "mode", "ok"]] = pd.DataFrame(res, index=df.index)

print("Images illisibles :", (~df.ok).sum())
print("Dimensions :", Counter(zip(df.w, df.h)).most_common(5))
print("Modes :", Counter(df["mode"]))

# Doublons exacts
dup = df[df.duplicated("md5", keep=False)]
print("\nImages dans un groupe de doublons exacts :", len(dup), "| groupes :", dup.md5.nunique())
cross = dup.groupby("md5").split.nunique()
print("Groupes exacts à cheval sur plusieurs partitions :", (cross > 1).sum())
lab_n = dup.groupby("md5").label.nunique()
print("Groupes exacts avec étiquettes contradictoires :", (lab_n > 1).sum())

# Sauvegarde de l'index sur le Drive
OUT = BASE / "index_bonefract.csv"
df.drop(columns=["path"]).to_csv(OUT, index=False)
print("\nIndex sauvegardé :", OUT)

### 2.2 Analyse des groupes de copies

In [ ]:
info = df.groupby("md5").agg(n=("md5", "size"), n_split=("split", "nunique"), n_label=("label", "nunique"),
             n_region=("region", "nunique"),
             splits=("split", lambda s: "+".join(sorted(set(s)))))
dupg = info[info.n > 1]

print("Taille des groupes :\n", dupg.n.value_counts().sort_index())
print("\nMulti-partition x étiquettes contradictoires :")
print(pd.crosstab(dupg.n_split > 1, dupg.n_label > 1,
                  rownames=["multi-partition"], colnames=["contradictoire"]))
print("\nPartitions concernées :\n", dupg.splits.value_counts())
print("\nGroupes sur plusieurs régions :", (dupg.n_region > 1).sum(), "/", len(dupg))

# Contamination : images de valid/test dont le md5 existe dans une autre partition
for s in ["valid", "test"]:
    others = set(df[df.split != s].md5)
    m = df.split == s
    c = df[m].md5.isin(others)
    print(f"\n{s} : {c.sum()} / {m.sum()} images ({c.mean():.1%}) ont un doublon exact dans une autre partition")
    print("   dont étiquette contradictoire avec l'autre partition :",
          sum(1 for _, r in df[m & df.md5.isin(others)].iterrows()
              if set(df[(df.md5 == r.md5) & (df.split != s)].label) - {r.label}))

# Les doublons sont-ils des images quasi uniformes (noires/blanches) ?
def stats(p):
    a = np.asarray(Image.open(p).convert("L"), dtype=np.float32)
    return a.mean(), a.std()
rep = df.drop_duplicates("md5").set_index("md5")
dup_paths = rep.loc[dupg.index, "path"].tolist()
non_paths = rep.drop(dupg.index).sample(3000, random_state=0).path.tolist()
with ThreadPoolExecutor(8) as ex:
    sd = np.array(list(ex.map(stats, dup_paths))); sn = np.array(list(ex.map(stats, non_paths)))
print("\nÉcart-type des pixels, groupes dupliqués :", np.percentile(sd[:, 1], [1, 5, 50]).round(1))
print("Écart-type des pixels, non dupliquées     :", np.percentile(sn[:, 1], [1, 5, 50]).round(1))

# Planche : 6 groupes contradictoires, 2 images chacun
bad = dupg[dupg.n_label > 1].sample(6, random_state=1).index
fig, axes = plt.subplots(6, 2, figsize=(7, 21))
for r, h in enumerate(bad):
    for c, (_, row) in enumerate(df[df.md5 == h].head(2).iterrows()):
        axes[r, c].imshow(Image.open(row.path).convert("L"), cmap="gray"); axes[r, c].axis("off")
        axes[r, c].set_title(f"{row.split} | {row.region} | label={row.label}", fontsize=8)
plt.tight_layout(); plt.show()

### 2.3 Politique de nettoyage et sauvegarde du pool

In [ ]:
# --- Anatomie des contradictions
cols = ["n", "n_split", "n_label", "n_region"]          # `info` vient de la cellule précédente
df = df.drop(columns=[c for c in cols if c in df.columns]).join(info[cols], on="md5")
c = info[info.n_label > 1]
print("Groupes contradictoires :", len(c))
print(pd.crosstab(c.n_region > 1, c.n_split > 1,
                  rownames=["régions différentes"], colnames=["partitions différentes"]))
print("Images concernées :", (df.n_label > 1).sum())

# --- Application de la politique
df["contradictory"] = df.n_label > 1
cand = df[~df.contradictory].assign(prio=lambda d: d.split.map(PRIO)).sort_values(["md5", "prio", "path"])
clean = cand.drop_duplicates("md5", keep="first").drop(columns="prio")
assert clean.md5.is_unique

removed = df[~df.path.isin(clean.path)].copy()
removed["reason"] = np.where(removed.contradictory, "etiquettes_contradictoires", "doublon_exact")
print("\nRetirées (contradictoires) :", (removed.reason == "etiquettes_contradictoires").sum())
print("Retirées (doublons exacts) :", (removed.reason == "doublon_exact").sum())
print("Pool nettoyé :", len(clean), "sur", len(df))

print("\nAvant / après par partition :")
print(pd.DataFrame({"avant": df.split.value_counts(), "après": clean.split.value_counts()}))
print("\nClasses après nettoyage :")
print(pd.crosstab(clean.split, clean.label, margins=True))
print("Prévalence fracture :", clean.groupby("split").label.mean().round(3).to_dict())
print("\nPar région et classe, après nettoyage :")
print(pd.crosstab([clean.split, clean.region], clean.label))

for s, n in NEED.items():
    have = (clean.split == s).sum()
    print(f"{s}: {have} disponibles / {n} requis ->", "OK" if have >= n else "INSUFFISANT")

# --- Sauvegarde (liste conservée pour l'annexe de l'article)
clean.drop(columns="path").to_csv(BASE / "pool_nettoye_bonefract.csv", index=False)
removed.drop(columns="path").to_csv(BASE / "images_retirees_bonefract.csv", index=False)

## 3. Quasi-doublons inter-partitions (corrélation de vignettes 32 × 32)

### 3.1 Vignettes du pool nettoyé

Chaque image est réduite à une vignette 32 × 32 en niveaux de gris, centrée puis normalisée en float64. Le résultat est mis en cache sur le Drive (`vignettes32_pool.npz`) ; si l'ordre du pool change, supprimer ce fichier.

In [ ]:
pool0 = clean.reset_index(drop=True)          # pool après contradictoires + doublons exacts
print("Pool MD5 :", len(pool0), "(attendu", EXPECTED_POOL_MD5, ")", pool0.split.value_counts().to_dict())
assert pool0.rel_path.is_unique and pool0.md5.is_unique

CACHE = BASE / "vignettes32_pool.npz"
if CACHE.exists():
    z = np.load(CACHE)
    assert (z["rel"] == pool0.rel_path.values.astype(str)).all(), "cache incompatible avec l'ordre du pool : supprime-le"
    X32 = z["X"]
else:
    with ThreadPoolExecutor(8) as ex:
        X32 = np.stack(list(ex.map(thumb, pool0.path, chunksize=64)))
    np.savez(CACHE, X=X32, rel=pool0.rel_path.values.astype(str))

X = X32.astype(np.float64)
nrm = np.linalg.norm(X, axis=1, keepdims=True)
X = X / np.maximum(nrm, 1e-12)
print("Vignettes :", X.shape, "| images à variance nulle :", int((nrm.ravel() < 1e-9).sum()))

sp = pool0.split.values
lab = pool0.label.values
pri = pool0.split.map(PRIO).values

### 3.2 Choix du seuil : plus proche voisin dans une autre partition

Pour chaque image de validation/test, meilleure corrélation avec une image d'une **autre** partition, puis inspection visuelle de paires par tranche de corrélation.

In [ ]:
tgt = np.where(sp != "train")[0]
best_c = np.full(len(pool0), -1.0, dtype=np.float32); best_j = np.full(len(pool0), -1)
for s in range(0, len(tgt), 500):
    idx = tgt[s:s + 500]
    S = X[idx] @ X.T
    S[sp[idx][:, None] == sp[None, :]] = -1
    j = S.argmax(1)
    best_c[idx] = S[np.arange(len(idx)), j]; best_j[idx] = j

print("Images valid/test dont le plus proche voisin (autre partition) dépasse le seuil :")
for t in [0.999, 0.995, 0.99, 0.98, 0.95, 0.90]:
    print(f"  corr >= {t}: valid={(best_c[sp == 'valid'] >= t).sum()}  test={(best_c[sp == 'test'] >= t).sum()}")

# Exemples par tranche de corrélation
bands = [(0.999, 1.01), (0.99, 0.999), (0.97, 0.99)]
picks = []
rng = np.random.RandomState(0)
for lo, hi in bands:
    c = tgt[(best_c[tgt] >= lo) & (best_c[tgt] < hi)]
    print(f"tranche [{lo}, {hi}) : {len(c)} images")
    picks += list(rng.choice(c, min(2, len(c)), replace=False)) if len(c) else []

nn_pairs = pd.DataFrame({"i": picks, "j": best_j[picks], "corr": best_c[picks]})
show_pairs(nn_pairs, pool0, "Plus proches voisins par tranche de corrélation", n=len(nn_pairs), head=True)

### 3.3 Application de la règle (seuil 0,97)

Paires inter-partitions uniques ≥ seuil : étiquettes différentes → retrait des deux images ; sinon retrait de la copie de la partition la moins prioritaire (test > validation > train).

In [ ]:
pairs, raw_thr = find_pairs(X, sp, LOW, THR)
pairs["combo"] = ["/".join(sorted([sp[i], sp[j]])) for i, j in zip(pairs.i, pairs.j)]
pairs["meme_etiquette"] = lab[pairs.i] == lab[pairs.j]
pairs_thr = pairs[pairs["corr"] >= THR]

print(f"Détections brutes >= {THR} (paires valid/test comptées 2 fois) : {raw_thr}")
print(f"Paires UNIQUES >= {THR} : {len(pairs_thr)}  -> doublons de comptage : {raw_thr - len(pairs_thr)}")
print("\nPaires uniques >= seuil par combinaison de partitions :")
print(pairs_thr.combo.value_counts().to_string())
print("\nPaires uniques >= seuil avec étiquettes différentes :", int((~pairs_thr.meme_etiquette).sum()))
print(f"Paires uniques dans [{LOW}, {THR}) (non retirées) :", int((pairs["corr"] < THR).sum()))

keep, drop = apply_rule(pairs_thr, lab, pri, len(pool0))
pool1 = pool0[keep].reset_index(drop=True)
X1 = X[keep]
print(f"\nImages retirées par la règle : {len(drop)} (annoncé au premier run : {EXPECTED_REMOVED_NEAR})")
print(pool0[~keep].groupby(["split", "label"]).size().to_string())
print(f"Pool final reconstruit : {len(pool1)} (annoncé : {EXPECTED_POOL_FINAL}) |", pool1.split.value_counts().to_dict())
if len(pool1) != EXPECTED_POOL_FINAL:
    print("NB : l'écart avec le premier run vient du passage en float64 (paires au voisinage immédiat du seuil).")

report["quasi_doublons"] = {"seuil": THR, "detections_brutes": int(raw_thr), "paires_uniques": int(len(pairs_thr)),
                            "par_combinaison": {k: int(v) for k, v in pairs_thr.combo.value_counts().items()},
                            "paires_etiquettes_differentes": int((~pairs_thr.meme_etiquette).sum()),
                            "images_retirees": int(len(drop)), "pool_final": int(len(pool1)),
                            "paires_uniques_sous_seuil_[%.2f,%.2f)" % (LOW, THR): int((pairs["corr"] < THR).sum())}
pairs.assign(rel_i=pool0.rel_path.values[pairs.i], rel_j=pool0.rel_path.values[pairs.j]).to_csv(BASE / "paires_quasi_doublons.csv", index=False)

### 3.4 Vérification automatique

In [ ]:
# Vérification automatique en float64 : plus aucune paire inter-partitions >= seuil dans le pool final
mx, n_over, arg = max_cross_corr(X1, pool1.split.values, THR)
print(f"Corrélation maximale restante entre partitions (pool final) : {mx:.8f}   (seuil {THR})")
print("Détections restantes >= seuil :", n_over)
assert n_over == 0 and mx < THR, "des paires inter-partitions dépassent encore le seuil"
print("OK : aucune paire inter-partitions n'atteint le seuil.")
report["verification_pool_final"] = {"correlation_max_restante": mx, "detections_au_dessus_du_seuil": int(n_over)}

### 3.5 Inspection visuelle autour du seuil

Paires juste au-dessus du seuil (retirées), puis juste en dessous (conservées), pour vérifier que 0,97 sépare bien copies et simples ressemblances de cadrage.

In [ ]:
above = pairs[(pairs["corr"] >= THR) & (pairs["corr"] < 0.98)]
below1 = pairs[(pairs["corr"] >= 0.96) & (pairs["corr"] < THR)]
below2 = pairs[(pairs["corr"] >= LOW) & (pairs["corr"] < 0.96)]
print("Effectifs :", f"[0.97,0.98): {len(above)} | [0.96,0.97): {len(below1)} | [0.95,0.96): {len(below2)}")
show_pairs(above, pool0, "Juste AU-DESSUS du seuil [0.97, 0.98) : retirées")
show_pairs(below1, pool0, "Juste EN DESSOUS du seuil [0.96, 0.97) : conservées")
show_pairs(below2, pool0, "Plus bas [0.95, 0.96) : conservées")

## 4. Tirage de l'échantillon final

Stratification région × classe à l'intérieur de chaque partition officielle, graine d'échantillonnage 42, tirage unique.

In [ ]:
sample = pd.concat([stratified_sample(pool1[pool1.split == s], n, SAMPLING_SEED) for s, n in NEED.items()])
sample = sample.sort_values(["split", "path"]).reset_index(drop=True)

# --- Contrôles
print("Effectifs :", sample.split.value_counts().to_dict(), "| total :", len(sample))
assert all(int((sample.split == s).sum()) == n for s, n in NEED.items()), "effectifs différents des cibles"
print("Prévalence fracture échantillon :", sample.groupby("split").label.mean().round(3).to_dict())
print("Prévalence fracture pool        :", pool1.groupby("split").label.mean().round(3).to_dict())
for s in NEED:
    a = pd.crosstab(sample[sample.split == s].region, sample[sample.split == s].label, normalize=True)
    b = pd.crosstab(pool1[pool1.split == s].region, pool1[pool1.split == s].label, normalize=True)
    print(f"  écart max région x classe {s} : {((a - b).abs().max().max() * 100):.2f} pt")
for col in ["md5", "patient_key"]:
    sets = {s: set(sample[sample.split == s][col]) for s in NEED}
    print(f"Chevauchement {col} :", {f"{a}/{b}": len(sets[a] & sets[b]) for a in NEED for b in NEED if a < b})
assert sample.md5.is_unique and sample.path.is_unique

In [ ]:
# --- Sauvegarde reproductible (sans écraser un tirage déjà enregistré s'il diffère)
cols_out = ["rel_path", "split", "region", "label", "md5", "patient"]
out_csv, out_cfg = SAMPLE_CSV, BASE / "echantillon_config.json"
if out_csv.exists():
    old = pd.read_csv(out_csv)
    if set(old.rel_path) != set(sample.rel_path):
        out_csv, out_cfg = BASE / "echantillon_15000_bonefract_REFAIT.csv", BASE / "echantillon_config_REFAIT.json"
        print("ATTENTION : le nouveau tirage diffère de l'échantillon déjà enregistré ; l'original est conservé.")
    else:
        print("Le nouveau tirage est identique à l'échantillon déjà enregistré.")

sample[cols_out].to_csv(out_csv, index=False)
config = {"seed_echantillonnage": SAMPLING_SEED, "n": {k: int(v) for k, v in sample.split.value_counts().items()},
          "pool_final": int(len(pool1)), "seuil_correlation_vignettes_32x32": THR,
          "politique": "etiquettes contradictoires retirees (MD5); doublons exacts: copie test>valid>train; quasi-doublons inter-partitions (corr>=0.97): etiquettes differentes -> retrait des deux, sinon copie de la partition la moins prioritaire; stratification region x classe"}
with open(out_cfg, "w") as f:
    json.dump(config, f, indent=2)
print("\nSauvegardé :", out_csv, "|", out_cfg)

## 5. Vérification finale et rapport

### 5.1 Relecture de l'échantillon enregistré

In [ ]:
# Vérification sur l'échantillon final de 15 000 images
samp = pd.read_csv(SAMPLE_CSV)
print("Échantillon :", samp.split.value_counts().to_dict(), "| total", len(samp))
pos = {r: k for k, r in enumerate(pool0.rel_path)}
idx_s = np.array([pos[r] for r in samp.rel_path])           # KeyError si l'échantillon sort du pool
in_final = pool0.rel_path.iloc[idx_s].isin(set(pool1.rel_path)).values
print("Images de l'échantillon absentes du pool final reconstruit :", int((~in_final).sum()))

mx_s, over_s, arg_s = max_cross_corr(X[idx_s], samp.split.values, THR)
print(f"Corrélation maximale entre partitions dans l'échantillon : {mx_s:.8f}")
assert over_s == 0 and mx_s < THR, "l'échantillon contient une paire inter-partitions >= seuil"

for col in ["md5"]:
    sets = {s: set(samp[samp.split == s][col]) for s in ["train", "valid", "test"]}
    print(f"Chevauchement {col} :", {f"{a}/{b}": len(sets[a] & sets[b]) for a in sets for b in sets if a < b})
assert samp.rel_path.is_unique and samp.md5.is_unique
chk_s = samp.merge(df[["rel_path", "label"]].rename(columns={"label": "label_folder"}), on="rel_path")
assert (chk_s.label == chk_s.label_folder).all(), "étiquette de l'échantillon ≠ dossier"
print("Étiquettes de l'échantillon = dossiers : OK |", "prévalence :", samp.groupby("split").label.mean().round(3).to_dict())
report["verification_echantillon"] = {"correlation_max": mx_s, "absentes_du_pool_final_float64": int((~in_final).sum())}

### 5.2 Rapport JSON

In [ ]:
with open(BASE / "rapport_verification.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str)[:3500])
print("\nSauvegardé :", BASE / "rapport_verification.json")

### 5.3 Recherche d'un patient Problematique dans l'échantillon

Le patient 26944 presentais une incoherence une verification est mise en oeuvre pour voir si il fait partie de notre echantillon dans ce cas des mesure serait prise sinon si  le patient est absent rien n'est fait
, la cellule indique s'il se trouve dans le pool nettoyé ou sur le disque (retiré au nettoyage, ou simplement non tiré).

In [ ]:
PATIENT = "patient26944"

# Recherche sur la colonne patient ET sur le chemin
m = (samp.patient.astype(str).str.lower() == PATIENT.lower()) | samp.rel_path.str.contains(PATIENT, case=False, regex=False)
hit = samp[m]

if hit.empty:
    print(f"{PATIENT} : ABSENT de l'échantillon de {len(samp)} images")
    in_pool = pool0.rel_path.str.contains(PATIENT, case=False, regex=False)
    in_disk = df.rel_path.str.contains(PATIENT, case=False, regex=False)
    print("Dans le pool nettoyé :", int(in_pool.sum()), "image(s) | Sur le disque :", int(in_disk.sum()), "image(s)")
    display(df[in_disk][["rel_path", "split", "label"]])
else:
    print(f"{PATIENT} : PRÉSENT, {len(hit)} image(s) dans l'échantillon")
    print(hit.split.value_counts().to_dict())
    display(hit[["rel_path", "split", "region", "label"]])

## Annexe A. Essai abandonné : hash perceptuel (dhash 9 × 8)

Conservé pour mémoire : le dhash entre partitions retirait 787 images (712 train, 75 valid), mais l'inspection visuelle a montré qu'il s'agissait de radiographies **différentes** (fonds noirs et cadrages standards → collisions). Il n'est utilisé nulle part dans la suite du pipeline.

In [ ]:
# Combien d'images ont un dhash identique à une autre (sur l'index complet) ?
nd = df[df.duplicated("dhash", keep=False)]
print("Images avec dhash identique à une autre :", len(nd), "| groupes :", nd.dhash.nunique())
print("Groupes dhash à cheval sur plusieurs partitions :", (nd.groupby("dhash").split.nunique() > 1).sum())

# Application du filtre dhash sur le pool nettoyé : on garde la copie de la partition prioritaire
c2 = pool0.assign(prio=pool0.split.map(PRIO))
c2["minp"] = c2.groupby("dhash").prio.transform("min")
drop_dh = c2.prio > c2.minp
print("\nRetirées pour quasi-doublon inter-partitions (dhash) :", drop_dh.sum())
print(c2[drop_dh].split.value_counts().to_dict())
pool_dh = c2[~drop_dh].drop(columns=["prio", "minp"]).reset_index(drop=True)

# Inspection visuelle : paires retirées / conservées
rem = c2[drop_dh].sample(6, random_state=0)
fig, axes = plt.subplots(6, 2, figsize=(7, 21))
for r, (_, row) in enumerate(rem.iterrows()):
    keep_row = pool_dh[pool_dh.dhash == row.dhash].iloc[0]
    a = np.asarray(Image.open(row.path).convert("L").resize((128, 128)), dtype=np.float32)
    b = np.asarray(Image.open(keep_row.path).convert("L").resize((128, 128)), dtype=np.float32)
    print(f"paire {r}: différence moyenne de pixels = {np.abs(a - b).mean():.2f} (sur 255)")
    for c, (x, tag) in enumerate([(row, "retirée"), (keep_row, "conservée")]):
        axes[r, c].imshow(Image.open(x.path).convert("L"), cmap="gray"); axes[r, c].axis("off")
        axes[r, c].set_title(f"{tag} | {x.split} | {x.region} | label={x.label}", fontsize=8)
plt.tight_layout(); plt.show()